# NLBSE'27 Code Classification: evaluation on Colab T4

Evaluates our early-exit UniXcoder student on the public test split (`ds["test"]`, 30,655 rows) with the
measurement loop of the organizers' reference notebook (`CodeBERT_baseline.ipynb`) and computes the
submission score with their formula and constants. **Run on a T4 GPU** (Runtime > Change runtime type).

### Differences from the baseline notebook

The batch size, the 10 forward passes per batch under `torch.profiler(with_flops=True)`, the timed region,
the FLOPs accumulation (sum of `k.flops` / 1e9, summed over batches, divided by 10) and the score are
unchanged. The baseline's other settings are its own configuration; this model differs as follows:

| Setting | Baseline notebook | This notebook |
|---|---|---|
| Model | CodeBERT `RobertaForSequenceClassification`, 12 layers | One UniXcoder-based multi-label model: 6 layers, FFN 1536, one exit head per layer |
| Input | `Function`, RoBERTa tokenizer, max 512 tokens | `Function` only: whitespace collapsed, a flag token if `TODO/FIXME/HACK/XXX` occurs, UniXcoder encoder-only format, max 512 tokens |
| Padding | whole set tokenized at once with `padding=True` (one global length) | `PADDING` below: default `sorted` = rows sorted by length, each batch padded to its own longest row (dynamic padding). `global` reproduces the baseline's padding |
| Batch size | 600 | 600 |
| Precision | fp32 weights under CUDA autocast | fp16 weights under CUDA autocast |
| Decision | `sigmoid(logits) > 0.5` | `sigmoid(logits) > 0.5` on logits already shifted by per-label thresholds chosen on validation |
| Forward | every row through all layers | a row leaves after layer *k* when both labels are confidently decided (early exit) |

Tokenization and moving each batch to the GPU happen outside the timed region, as in the reference.
Every row is predicted once and predictions are written back in the original order; no sampling or
rebalancing is applied to the evaluation set.


In [ ]:
!pip -q install datasets safetensors

## 1. Configuration

In [ ]:
# Exported model: a zip on Google Drive (MyDrive/nlbse27/), or set MODEL_ZIP = None and
# MODEL_SOURCE to a Hugging Face repo id or a local folder.
MODEL_ZIP = "/content/drive/MyDrive/nlbse27/R8_6L_ffn1536_exits.zip"
MODEL_SOURCE = None

PADDING = "sorted"            # "sorted" (ours), "dynamic" (per batch, original order), "global" (as the baseline)
batch_size = 600
repetitions = 10
RUN_PADDING_ABLATION = False  # also measure all three padding modes (adds ~15-20 min on a T4)

In [ ]:
import sys, zipfile
from pathlib import Path
if MODEL_ZIP:
    from google.colab import drive
    drive.mount("/content/drive")
    zipfile.ZipFile(MODEL_ZIP).extractall("/content/model")
    MODEL_SOURCE = "/content/model"
if not Path(MODEL_SOURCE).exists():
    from huggingface_hub import snapshot_download
    MODEL_SOURCE = snapshot_download(MODEL_SOURCE)
sys.path.insert(0, MODEL_SOURCE)   # the export ships its own nlbse27ee package
print(sorted(p.name for p in Path(MODEL_SOURCE).iterdir()))

In [ ]:
import time, numpy as np, pandas as pd, torch
from datasets import load_dataset
from nlbse27ee.export import load_model
from nlbse27ee.data import Packed, inference_batches, pad_rows
from nlbse27ee.preprocess import pack

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu = torch.cuda.get_device_name(device)
print(gpu)
if "T4" not in gpu:
    print("WARNING: not a T4; runtime is not comparable with the official setting.")
model, encoder = load_model(MODEL_SOURCE, device)   # model.eval() inside
print(model.cfg, encoder.describe())

## 2. Data and preprocessing (outside the timed region)

In [ ]:
ds = load_dataset("NLBSE/nlbse27-code-comment-classification")
test = ds["test"]
assert len(test) == 30655, len(test)
test_data = test.to_pandas()
packed = Packed(*pack(encoder(test_data["Function"].tolist())))
pad_id = encoder.tokenizer.pad_token_id
print(len(packed), "rows; mean tokens", packed.lengths.mean().round(1), "; max", packed.lengths.max())

## 3. Inference with the reference measurement loop

In [ ]:
total_flops = 0
total_time = 0
padded_tokens = 0
predictions = np.full((len(packed), 2), -1, dtype=np.int64)

for rows, width in inference_batches(packed.lengths, batch_size, PADDING):
    input_ids, attention_mask = pad_rows(packed, rows, pad_id, width=width)
    padded_tokens += input_ids.numel()
    input_ids = input_ids.to(device, non_blocking=True)
    attention_mask = attention_mask.to(device, non_blocking=True)

    # Perform forward pass and measure performance
    begin = time.time()
    with torch.no_grad():
      with torch.amp.autocast("cuda"):
        with torch.profiler.profile(with_flops=True) as p:
            for _ in range(repetitions):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    total = time.time() - begin
    total_time = total_time + total

    # Calculate FLOPs for the batch
    total_flops = total_flops + (sum(k.flops for k in p.key_averages()) / 1e9)

    probs = torch.sigmoid(outputs.logits)
    threshold = 0.5
    predictions[rows] = (probs > threshold).int().cpu().numpy()
    del input_ids, attention_mask, outputs, probs
    torch.cuda.empty_cache()

assert predictions.shape == (30655, 2) and (predictions >= 0).all(), "every row must be predicted"
print(f"padding={PADDING}: {padded_tokens:,} padded tokens for {int(packed.lengths.sum()):,} real tokens")

## 4. Metrics and submission score (reference code)

In [ ]:
labels = np.array(test_data["Label"].tolist())
labels_name = ["Vulnerability", "MAT"]
scores = []
for class_idx in range(len(labels_name)):
    tp = np.sum((predictions[:, class_idx] == 1) & (labels[:, class_idx] == 1))
    fp = np.sum((predictions[:, class_idx] == 1) & (labels[:, class_idx] == 0))
    fn = np.sum((predictions[:, class_idx] == 0) & (labels[:, class_idx] == 1))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    scores.append({"cat": labels_name[class_idx], "precision": precision, "recall": recall, "f1": f1})
scores = pd.DataFrame(scores)

max_avg_runtime = 1000
max_avg_flops = 3000000
def score(avg_f1, avg_runtime, avg_flops):
    return (0.6 * avg_f1
            + 0.2 * max((max_avg_runtime - avg_runtime) / max_avg_runtime, 0)
            + 0.2 * max((max_avg_flops - avg_flops) / max_avg_flops, 0))

avg_f1, avg_p, avg_r = scores.f1.mean(), scores.precision.mean(), scores.recall.mean()
avg_runtime, avg_flops = total_time / repetitions, total_flops / repetitions
print("GPU:", gpu, "| padding:", PADDING, "| batch size:", batch_size, "| repetitions:", repetitions)
print("Compute in GFLOPs (one pass over the set):", avg_flops)
print("Avg runtime in seconds (one pass over the set):", avg_runtime)
print("Mean precision / recall / F1:", avg_p, avg_r, avg_f1)
print("Submission score:", round(score(avg_f1, avg_runtime, avg_flops), 4), "(CodeBERT baseline 0.7299)")
scores

## 5. Optional: padding ablation
Same loop for `global` (baseline padding), `dynamic` and `sorted`. Shows how much of the cost reduction comes from dynamic padding versus the model.

In [ ]:
if RUN_PADDING_ABLATION:
    from IPython.display import Markdown, display
    from nlbse27ee.padding_ablation import run, markdown
    display(Markdown(markdown(run(MODEL_SOURCE, batch_size, repetitions))))

## 6. Predicting new data (e.g. the hidden test)
`predict_labels` returns `[Vulnerable, MAT]` labels in input order.

In [ ]:
from nlbse27ee.evaluate import predict_labels
predict_labels(model, encoder, test_data["Function"].tolist()[:5], device)